# MoveScope Ablation Experiment

Compares 2D, 3D, weighted DTW, and full weighted segmented DTW variants on good and bad squat videos.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from movescope.experiments import run_ablation, summarize_ablation, run_viewpoint_consistency, viewpoint_std
from movescope.template import ActionTemplate

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
template_path = Path("data/templates/squat.npz")
if not template_path.exists():
    print("Missing data/templates/squat.npz. Build it with scripts/build_template.py before running this experiment.")
    rows = []
else:
    template = ActionTemplate.load("squat")
    rows = run_ablation(template, good_dir="data/test/good_squat", bad_dir="data/test/bad_squat")
    print(f"Collected {len(rows)} scored rows")


In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    print("No scored videos found. Populate data/test/good_squat and data/test/bad_squat first.")
else:
    display(df.head())
    display(pd.DataFrame(summarize_ablation(rows)))


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    df.boxplot(column="total_score", by=["variant", "label"], ax=ax, rot=30)
    ax.set_title("Good vs bad score distribution by variant")
    ax.set_ylabel("total_score")
    fig.suptitle("")
    fig.tight_layout()
    output = FIG_DIR / "ablation_result.png"
    fig.savefig(output, dpi=160)
    print(f"Saved {output}")


In [ ]:
if not df.empty:
    for variant in sorted(df["variant"].unique()):
        good = df[(df.variant == variant) & (df.label == "good")]["total_score"]
        bad = df[(df.variant == variant) & (df.label == "bad")]["total_score"]
        if len(good) >= 2 and len(bad) >= 2:
            test = stats.ttest_ind(good, bad, equal_var=False)
            print(f"{variant}: separation={good.mean() - bad.mean():.2f}, p={test.pvalue:.4g}")
        else:
            print(f"{variant}: need at least 2 good and 2 bad videos for a t-test")
